<a href="https://colab.research.google.com/github/Rohith715/Natural_Language_Processing/blob/main/RAG_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#this simple project aims to create a simple pipeline to process any pdf sent my way and make a rag chatbot out of it
#turns out there is a super accurate rag system better than langchain+pinecone /lamaindex called groundx
#[phewww] there is a lot going on by=ut for now les jus make a simple rag using Pypdf2 and pine cone and a generative ai



#steps involved in creating a rag system ----
#---1> parse information
#---2> embed informnation
#---3> feed it to a vector database
#---4> take a prompt and embed it and send it to vector db for querying
#---5> take the answer from db and generate an answer using gen ai again


#[update ---]->> while i was working with the parsing part
#i understood the importance of chunking the text into a meaningful paragraphs to do this i am gonna use

In [4]:
!pip install pypdf2==3.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.4/232.4 kB 3.8 MB/s eta 0:00:00


In [5]:
from PyPDF2 import PdfReader
read=PdfReader('/content/CV Rohith -12208052--done.pdf')
gg=[]
for x in range(len(read.pages)):
  page=read.pages[x]
  text = page.extract_text()
  gg.append(text)


In [6]:
i=2
for x in gg:
  print(x,end=f"###--pager-- {i}")
  i=i+1

Paidimuddala Rohith Yadav  
Linkedin:  https://www.linkedin.com/in/phoenix715/  Email:  marl9176327179@gmail.com  
Github:  https://github.com/Rohith715  Mobile:   +91-8360867137  
 
SKILLS  
• Languages : C++,  JavaScript,  C, Java,  Python  
• Frameworks : HTML  and CSS,  Scikit -learn, Num Py, Pandas, Matplotlib  
• Tools/Platforms : Tableau , Automation Anywhere, Blender  
• Soft  Skills : Problem -Solving  Skills,  Team  Player,  Project  Management,  Adaptability , Communication  
 
PROJECTS  
Text Auto -complete model :          August 2024  
• Aim : To design a language model that helps with autocomplete  
• Description:  This language model learns from a given text corpus and utilizes techniques like N-gram  to 
predict the next possible wordings suitable for text completion.  
• Tech:  Python, Pandas, Scikit -learn, NumPy,  Seaborn,  Matplotlib , N-grams , Natural Language 
Processing . 
 Text Generator :           July 2024  
• Aim:  To create a text generator that creates t

In [ ]:
from google import genai
from PIL import Image
API_KEY=''
prompt="process the provided text and space seperate them according to their respective caterories CONTACT,SKILLS,PROJECTS,CERTIFICATES,ACHIEVEMENTS,EDUCATION and return the text end each section with a #########"
def callgenai(API_KEY,prompt_giv,information):
  client = genai.Client(api_key=API_KEY)
  prompt=prompt_giv,
  response = client.models.generate_content(
      model="gemini-1.5-pro",
      contents=[information,prompt],
  )
  return response
response=callgenai(API_KEY,prompt,gg)

In [12]:
check=[]
for text in response.text.split("#########"):
  check.append(text)
order=["CONTACT","SKILLS","PROJECTS","CERTIFICATES","ACHIEVEMENTS","EDUCATION"]
##this above is the order in which the text will always be arranged as this will be given to the generative ai

In [14]:
!pip install --upgrade pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 6.4 MB/s eta 0:00:00


In [15]:
kk = []
for i, text_chunk in enumerate(check):
  if text_chunk!="\n":
    kk.append({
        "id": f"vec{i}",  # Unique ID for each chunk
        "text": text_chunk  # Text content of the chunk
    })

In [16]:
for x in kk:
  print(x)

{'id': 'vec0', 'text': '**CONTACT**\nPaidimuddala Rohith Yadav\nLinkedin: https://www.linkedin.com/in/phoenix715/\nEmail: marl9176327179@gmail.com\nGithub: https://github.com/Rohith715\nMobile: +91-8360867137\n'}
{'id': 'vec1', 'text': '\n\n**SKILLS**\nLanguages: C++ JavaScript C Java Python\nFrameworks: HTML CSS Scikit-learn NumPy Pandas Matplotlib\nTools/Platforms: Tableau Automation Anywhere Blender\nSoft Skills: Problem-Solving Skills Team Player Project Management Adaptability Communication\n'}
{'id': 'vec2', 'text': '\n\n**PROJECTS**\nText Auto-complete model: August 2024\nAim: To design a language model that helps with autocomplete\nDescription: This language model learns from a given text corpus and utilizes techniques like N-gram to predict the next possible wordings suitable for text completion.\nTech: Python Pandas Scikit-learn NumPy Seaborn Matplotlib N-grams Natural Language Processing\n\nText Generator: July 2024\nAim: To create a text generator that creates text similar 

In [17]:
#use the below code to create a pine cone index in your pine cone server, if u havent set it up already,
# before this u should have created an vector database on pine cone as a req
#index_name = "dense-index"
#if not pc.has_index(index_name):
#    pc.create_index_for_model(
#        name=index_name,
#        cloud="aws",
#        region="us-east-1",
#        embed={
#            "model":"llama-text-embed-v2",
#            "field_map":{"text": "chunk_text"}
#        }
#    )
#

from pinecone import Pinecone
PINE_API=""
pc = Pinecone(PINE_API)
index = pc.Index("text-rag")
# Embed data
data = kk

embeddings = pc.inference.embed(
    model="llama-text-embed-v2",
    inputs=[d['text'] for d in data],
    parameters={
        "input_type": "passage"
    }
)

vectors = []
for d, e in zip(data, embeddings):
    vectors.append({
        "id": d['id'],
        "values": e['values'],
        "metadata": {'text': d['text']}
    })

index.upsert(
    vectors=vectors,
    namespace="one"
)

{'upserted_count': 6}

In [ ]:
query = "talk about your most recent achievements and what you learnt from them "

x = pc.inference.embed(
    model="llama-text-embed-v2",
    inputs=[query],
    parameters={
        "input_type": "query"
    }
)

results = index.query(
    namespace="one",
    vector=x[0].values,
    top_k=3,
    include_values=False,
    include_metadata=True
)

print(results)

In [ ]:
prompt=query+"use the provided information to reply"
client = genai.Client(api_key=API_KEY)
# Convert results and prompt to dictionaries with 'text' key
contents = [{'text': str(results)}, {'text': prompt}]
response = client.models.generate_content(
    model="gemini-1.5-pro",
    contents=contents,
)
print(response.text)